# Majority Voting untuk Dataset Final

Notebook ini melakukan:
1. **Case 1 (Konflik)**: Jika semua 3 annotator berbeda dalam category ATAU sentiment → beri label konflik dan hapus
2. **Case 2 (Majority Voting)**: Jika 2 dari 3 annotator setuju → pilih label mayoritas (tidak ada konflik)
3. Simpan kedua dataset (dengan konflik dan final tanpa konflik)
4. Analisis perbedaan: per baris, per category, per sentiment

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Load data dari 3 annotator
df1 = pd.read_csv('very_clean_dataset/annotator1_clean.csv')
df2 = pd.read_csv('very_clean_dataset/annotator2_clean.csv')
df3 = pd.read_csv('very_clean_dataset/annotator3_clean.csv')

print(f"Annotator 1: {len(df1)} rows")
print(f"Annotator 2: {len(df2)} rows")
print(f"Annotator 3: {len(df3)} rows")
print("\nSample dari Annotator 1:")
df1.head()

In [ ]:
# Merge data dari 3 annotator berdasarkan sentence_id
# Rename kolom untuk membedakan masing-masing annotator
df1_renamed = df1.rename(columns={'category': 'category_1', 'sentiment': 'sentiment_1'})
df2_renamed = df2.rename(columns={'category': 'category_2', 'sentiment': 'sentiment_2'})
df3_renamed = df3.rename(columns={'category': 'category_3', 'sentiment': 'sentiment_3'})

# Merge berdasarkan sentence_id, at, content
merged = df1_renamed[['sentence_id', 'at', 'content', 'category_1', 'sentiment_1']].merge(
    df2_renamed[['sentence_id', 'category_2', 'sentiment_2']], 
    on='sentence_id', 
    how='inner'
).merge(
    df3_renamed[['sentence_id', 'category_3', 'sentiment_3']], 
    on='sentence_id', 
    how='inner'
)

print(f"Merged dataset: {len(merged)} rows")
print("\nSample merged data:")
merged.head(10)

In [ ]:
# Fungsi untuk majority voting
def get_majority_vote(values):
    """
    Mengembalikan nilai mayoritas dari list.
    Jika semua berbeda, return None (konflik).
    Jika ada 2 yang sama, return nilai mayoritas.
    """
    counter = Counter(values)
    most_common = counter.most_common(1)[0]
    
    # Jika nilai terbanyak muncul >= 2 kali, itu mayoritas
    if most_common[1] >= 2:
        return most_common[0]
    else:
        # Semua berbeda (konflik)
        return None

# Apply majority voting untuk category dan sentiment
merged['category_final'] = merged.apply(
    lambda row: get_majority_vote([row['category_1'], row['category_2'], row['category_3']]), 
    axis=1
)

merged['sentiment_final'] = merged.apply(
    lambda row: get_majority_vote([row['sentiment_1'], row['sentiment_2'], row['sentiment_3']]), 
    axis=1
)

# Tandai konflik
# Konflik = category_final ATAU sentiment_final adalah None
merged['is_conflict'] = (merged['category_final'].isna()) | (merged['sentiment_final'].isna())

print(f"Total data: {len(merged)}")
print(f"Data dengan konflik: {merged['is_conflict'].sum()}")
print(f"Data tanpa konflik (majority voting berhasil): {(~merged['is_conflict']).sum()}")
print("\nPersentase konflik: {:.2f}%".format(merged['is_conflict'].sum() / len(merged) * 100))

In [ ]:
# Tampilkan sample data konflik
print("="final * 80)
print("SAMPLE DATA KONFLIK (Semua 3 annotator berbeda):")
print("=" * 80)
conflict_data = merged[merged['is_conflict'] == True]
if len(conflict_data) > 0:
    display(conflict_data[['sentence_id', 'content', 
                           'category_1', 'category_2', 'category_3', 'category_final',
                           'sentiment_1', 'sentiment_2', 'sentiment_3', 'sentiment_final']].head(10))
else:
    print("Tidak ada data konflik!")

print("\n" + "=" * 80)
print("SAMPLE DATA MAJORITY VOTING (2 dari 3 annotator setuju):")
print("=" * 80)
non_conflict_data = merged[merged['is_conflict'] == False]
if len(non_conflict_data) > 0:
    display(non_conflict_data[['sentence_id', 'content', 
                                'category_1', 'category_2', 'category_3', 'category_final',
                                'sentiment_1', 'sentiment_2', 'sentiment_3', 'sentiment_final']].head(10))
else:
    print("Tidak ada data majority voting!")

In [ ]:
# Simpan dataset dengan konflik (untuk referensi)
dataset_with_conflict = merged[['sentence_id', 'at', 'content', 
                                 'category_1', 'category_2', 'category_3', 'category_final',
                                 'sentiment_1', 'sentiment_2', 'sentiment_3', 'sentiment_final',
                                 'is_conflict']].copy()

dataset_with_conflict.to_csv('dataset_with_conflict.csv', index=False)
print(f"✓ Dataset dengan label konflik disimpan: dataset_with_conflict.csv ({len(dataset_with_conflict)} rows)")

# Simpan dataset final (tanpa konflik - hanya majority voting)
dataset_final = merged[merged['is_conflict'] == False][['sentence_id', 'at', 'content', 
                                                          'category_final', 'sentiment_final']].copy()
dataset_final = dataset_final.rename(columns={'category_final': 'category', 'sentiment_final': 'sentiment'})

dataset_final.to_csv('dataset_final_majority_voting.csv', index=False)
print(f"✓ Dataset final (tanpa konflik) disimpan: dataset_final_majority_voting.csv ({len(dataset_final)} rows)")

print(f"\nData yang dihapus (konflik): {len(dataset_with_conflict) - len(dataset_final)} rows")

## Analisis Perbedaan Data

Bandingkan perbedaan antara:
- **Dataset dengan konflik** (semua data termasuk yang konflik)
- **Dataset final** (hanya data majority voting, konflik dihapus)

In [ ]:
# 1. PERBEDAAN PER BARIS
print("=" * 80)
print("1. PERBEDAAN JUMLAH BARIS")
print("=" * 80)

jumlah_total = len(dataset_with_conflict)
jumlah_final = len(dataset_final)
jumlah_konflik = dataset_with_conflict['is_conflict'].sum()

print(f"Dataset dengan konflik (semua data): {jumlah_total:,} rows")
print(f"Dataset final (tanpa konflik):       {jumlah_final:,} rows")
print(f"Data yang dihapus (konflik):         {jumlah_konflik:,} rows")
print(f"\nPersentase data yang dipertahankan: {jumlah_final/jumlah_total*100:.2f}%")
print(f"Persentase data yang dihapus:       {jumlah_konflik/jumlah_total*100:.2f}%")

In [ ]:
# 2. PERBEDAAN PER CATEGORY
print("\n" + "=" * 80)
print("2. PERBEDAAN DISTRIBUSI CATEGORY")
print("=" * 80)

# Category di dataset dengan konflik (gunakan category_final, skip yang None)
category_with_conflict = dataset_with_conflict[dataset_with_conflict['category_final'].notna()]['category_final'].value_counts().sort_index()

# Category di dataset final
category_final = dataset_final['category'].value_counts().sort_index()

# Gabungkan untuk perbandingan
comparison_category = pd.DataFrame({
    'Dengan Konflik (non-null)': category_with_conflict,
    'Final (tanpa konflik)': category_final
}).fillna(0).astype(int)

comparison_category['Selisih'] = comparison_category['Dengan Konflik (non-null)'] - comparison_category['Final (tanpa konflik)']
comparison_category['% Perubahan'] = (comparison_category['Selisih'] / comparison_category['Dengan Konflik (non-null)'] * 100).round(2)

print("\nDistribusi Category:")
print(comparison_category)

# Tampilkan juga yang konflik category (semua 3 berbeda)
category_conflict_count = dataset_with_conflict[dataset_with_conflict['category_final'].isna()].shape[0]
print(f"\nData dengan konflik category (3 annotator berbeda): {category_conflict_count}")

In [ ]:
# 3. PERBEDAAN PER SENTIMENT
print("\n" + "=" * 80)
print("3. PERBEDAAN DISTRIBUSI SENTIMENT")
print("=" * 80)

# Sentiment di dataset dengan konflik (gunakan sentiment_final, skip yang None)
sentiment_with_conflict = dataset_with_conflict[dataset_with_conflict['sentiment_final'].notna()]['sentiment_final'].value_counts().sort_index()

# Sentiment di dataset final
sentiment_final = dataset_final['sentiment'].value_counts().sort_index()

# Gabungkan untuk perbandingan
comparison_sentiment = pd.DataFrame({
    'Dengan Konflik (non-null)': sentiment_with_conflict,
    'Final (tanpa konflik)': sentiment_final
}).fillna(0).astype(int)

comparison_sentiment['Selisih'] = comparison_sentiment['Dengan Konflik (non-null)'] - comparison_sentiment['Final (tanpa konflik)']
comparison_sentiment['% Perubahan'] = (comparison_sentiment['Selisih'] / comparison_sentiment['Dengan Konflik (non-null)'] * 100).round(2)

print("\nDistribusi Sentiment:")
print(comparison_sentiment)

# Tampilkan juga yang konflik sentiment (semua 3 berbeda)
sentiment_conflict_count = dataset_with_conflict[dataset_with_conflict['sentiment_final'].isna()].shape[0]
print(f"\nData dengan konflik sentiment (3 annotator berbeda): {sentiment_conflict_count}")

In [ ]:
# 4. VISUALISASI PERBANDINGAN
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Jumlah Total Data
ax1 = axes[0, 0]
categories_bar = ['Dataset dengan Konflik', 'Dataset Final', 'Data Dihapus (Konflik)']
values_bar = [jumlah_total, jumlah_final, jumlah_konflik]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c']
ax1.bar(categories_bar, values_bar, color=colors_bar)
ax1.set_ylabel('Jumlah Baris')
ax1.set_title('Perbandingan Jumlah Data')
for i, v in enumerate(values_bar):
    ax1.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

# Plot 2: Distribusi Category
ax2 = axes[0, 1]
comparison_category[['Dengan Konflik (non-null)', 'Final (tanpa konflik)']].plot(kind='bar', ax=ax2, color=['#3498db', '#2ecc71'])
ax2.set_ylabel('Jumlah')
ax2.set_title('Distribusi Category: Dengan vs Tanpa Konflik')
ax2.legend(['Dengan Konflik', 'Final'])
ax2.set_xlabel('Category')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 3: Distribusi Sentiment
ax3 = axes[1, 0]
comparison_sentiment[['Dengan Konflik (non-null)', 'Final (tanpa konflik)']].plot(kind='bar', ax=ax3, color=['#3498db', '#2ecc71'])
ax3.set_ylabel('Jumlah')
ax3.set_title('Distribusi Sentiment: Dengan vs Tanpa Konflik')
ax3.legend(['Dengan Konflik', 'Final'])
ax3.set_xlabel('Sentiment')
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 4: Pie chart - Data Retention
ax4 = axes[1, 1]
pie_labels = [f'Dipertahankan\n({jumlah_final:,})', f'Dihapus (Konflik)\n({jumlah_konflik:,})']
pie_values = [jumlah_final, jumlah_konflik]
pie_colors = ['#2ecc71', '#e74c3c']
ax4.pie(pie_values, labels=pie_labels, autopct='%1.1f%%', colors=pie_colors, startangle=90)
ax4.set_title('Proporsi Data yang Dipertahankan vs Dihapus')

plt.tight_layout()
plt.savefig('majority_voting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualisasi disimpan: majority_voting_comparison.png")

In [ ]:
# RINGKASAN AKHIR
print("=" * 80)
print("RINGKASAN HASIL MAJORITY VOTING")
print("=" * 80)
print("\n📊 FILE OUTPUT:")
print("   1. dataset_with_conflict.csv          - Dataset lengkap dengan label konflik")
print("   2. dataset_final_majority_voting.csv  - Dataset final (hanya majority voting)")
print("   3. majority_voting_comparison.png     - Visualisasi perbandingan")

print("\n📈 STATISTIK:")
print(f"   • Total data awal: {jumlah_total:,} rows")
print(f"   • Data final (dipertahankan): {jumlah_final:,} rows ({jumlah_final/jumlah_total*100:.2f}%)")
print(f"   • Data konflik (dihapus): {jumlah_konflik:,} rows ({jumlah_konflik/jumlah_total*100:.2f}%)")

print("\n🎯 CASE YANG DITERAPKAN:")
print("   • Case 1: Konflik (semua 3 annotator berbeda) → Label konflik & dihapus")
print("   • Case 2: Majority voting (2 dari 3 setuju) → Pilih label mayoritas")

print("\n✅ Selesai!")